# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

✅ EmbeddingManager initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[LCBD] PEAK with Requirements", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [LCBD] PEAK with Requirements, Similarity: 0.75, Type: Diagram, Phase: Logical Architecture LA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
#import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## ⚙️ Execute  Prompt


Execute the prompt that will use the model and system document. 

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)
prompt = """
Write image description for the visually impaired based on the content of this diagram. 
Specifically call attention to how the PEAK System is decomposed
-Describe requirements related to each subsystem
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

✅ ChatGPTAnalyzer initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o


**Your prompt:** 
Write image description for the visually impaired based on the content of this diagram. 
Specifically call attention to how the PEAK System is decomposed
-Describe requirements related to each subsystem


**Response:**

The diagram represents a complex system model, specifically focusing on the PEAK System and its decomposition into various subsystems. The PEAK System is depicted as a central entity, surrounded by its constituent subsystems, each with specific components and functions. Here's a detailed description:

1. **PEAK System Overview**:
   - The PEAK System is a comprehensive setup designed to provide essential services during crises. It integrates multiple subsystems, each contributing to the overall functionality and efficiency of the system.

2. **Subsystems and Their Decomposition**:
   - **Water Subsystem**:
     - Composed of components like Water Purification, Water Filtration, Storage Container, and Controls System.
     - Requirements: Capable of purifying fresh, brackish, or salt water, integrated with power generation. The storage container must meet durability and lightweight standards.
   
   - **Power Sub-System**:
     - Includes Renewable Power and Fossil Fuel Generator Backup.
     - Requirements: Must power all kit components, providing reliability in diverse conditions where renewable sources may be insufficient. The subsystem should occupy no more than 19% of the total PEAK volume, equating to 100.58 cubic feet.
   
   - **Communication Subsystem**:
     - Consists of a Communication Device and a Distributed Communication Network.
     - Requirements: Integration with situational awareness components, facilitating mobile and decentralized communications. The subsystem should not exceed 29% of the total PEAK volume, approximately 100.58 cubic feet.
   
   - **Situational Awareness Sub-System**:
     - Comprises an Unmanned Air Vehicle and a Control Device and Platform.
     - Requirements: Provides an aerial perspective for quick assessment, integrating with the power system. The subsystem should occupy no more than 19% of the total PEAK volume, approximately 47.65 cubic feet.
   
   - **Pre-Position Kit Packaging**:
     - Focuses on functions like Receive Transport and Accept Storage.
     - Requirements: The packaging should not exceed 4% of the total PEAK volume, which is 21.17 cubic feet.

3. **Functional and Component Exchanges**:
   - The diagram illustrates various functional exchanges between components, such as water purification and storage, power provision, and communication management. These exchanges ensure the seamless operation of the PEAK System, highlighting the interdependencies between subsystems.

4. **Requirements and Constraints**:
   - The system must conform to specific dimensional constraints to ensure compatibility with standard transport and storage solutions. The overall volume of the system is approximately 529.36 cubic feet.

This diagram provides a comprehensive view of the PEAK System's architecture, emphasizing the decomposition into subsystems, each with distinct roles and requirements, ensuring efficient crisis response and management.

**Token Usage Info:**

Tokens used: prompt=29348, completion=561, total=29909

## 💬 Launch Interactive Chat on Structured Input




In [6]:
print("Done")

Done


# 